<a href="https://colab.research.google.com/github/Shorovpaul/Data_Testing/blob/main/kidney_disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas numpy scikit-learn imbalanced-learn shap xgboost

In [6]:
import pandas as pd

data = pd.read_csv("kidney_disease.csv")

print(data.shape)
# Step 1: Rename the column
data.rename(columns={'classification': 'Class'}, inplace=True)

# Step 2: Verify it worked
print(data.columns.tolist())

# Step 3: Clean the 'Class' column (target variable)
data['Class'] = data['Class'].replace({'ckd\t': 'ckd'})
print(data['Class'].value_counts())

(400, 26)
['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'Class']
Class
ckd       250
notckd    150
Name: count, dtype: int64


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Split features and target
X = data.drop("Class", axis=1)
y = data["Class"]

# Encode target variable
le = LabelEncoder()
y = le.fit_transform(y)

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [29]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Ensure y_train is numerical before SMOTE, in case previous steps didn't propagate
# This is a redundant step if gJqMEZtHgOEY was executed correctly, but ensures robustness
if y_train.dtype == 'object':
    le_smote = LabelEncoder()
    y_train_encoded = le_smote.fit_transform(y_train)
else:
    y_train_encoded = y_train

smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train_encoded)

# Convert y_train_bal back to a Series for consistent value_counts display, if it's an ndarray
if isinstance(y_train_bal, np.ndarray):
    y_train_bal = pd.Series(y_train_bal, name='Class')

print(y_train_bal.value_counts())

Class
1    200
0    200
Name: count, dtype: int64


In [30]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_bal = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import xgboost as xgb

models = {
    "LR": LogisticRegression(max_iter=1000),
    "RF": RandomForestClassifier(n_estimators=200),
    "GB": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True),
    "XGB": xgb.XGBClassifier(eval_metric="logloss")
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    trained_models[name] = model

In [32]:
from sklearn.metrics import average_precision_score

model_scores = {}

for name, model in trained_models.items():
    probs = model.predict_proba(X_test_scaled)[:,1]
    score = average_precision_score(y_test, probs)
    model_scores[name] = score

print(model_scores)

{'LR': np.float64(0.9978136200716845), 'RF': np.float64(0.9999999999999999), 'GB': np.float64(1.0), 'KNN': np.float64(0.8558683872409363), 'SVM': np.float64(0.9597423412588726), 'XGB': np.float64(0.9989247311827957)}


In [17]:
selected_models = {
    name:model for name,model in trained_models.items()
    if model_scores[name] > 0.70
}

In [24]:
import numpy as np
import pandas as pd

# Drop the 'id' column as it's not a feature
data.drop("id", axis=1, inplace=True, errors='ignore')

# Separate numerical and categorical columns
numerical_cols = data.select_dtypes(include=np.number).columns.tolist()
categorical_cols = data.select_dtypes(include='object').columns.tolist()
if 'Class' in categorical_cols:
    categorical_cols.remove('Class') # Target variable will be handled separately

# Handle 'dirty' numerical columns (pcv, wc, rc) - convert to numeric and impute
dirty_numerical_cols = ['pcv', 'wc', 'rc']
for col in dirty_numerical_cols:
    if col in data.columns:
        # Convert to numeric, coercing errors will turn invalid parsing into NaN
        data[col] = pd.to_numeric(data[col], errors='coerce')
        # Impute missing values with the median
        if data[col].isnull().any():
            data[col].fillna(data[col].median(), inplace=True)

# Impute other numerical columns with median
for col in numerical_cols:
    if col in data.columns and data[col].isnull().any():
        data[col].fillna(data[col].median(), inplace=True)

# Clean and impute categorical columns with mode
for col in categorical_cols:
    if col in data.columns:
        # Clean by stripping whitespace and lowercasing
        data[col] = data[col].astype(str).str.strip().str.lower()
        # Replace 'nan' string representation with actual NaN for proper imputation
        data[col].replace('nan', np.nan, inplace=True)
        # Impute missing values with the mode
        if data[col].isnull().any():
            mode_val = data[col].mode()[0]
            data[col].fillna(mode_val, inplace=True)

# Perform One-Hot Encoding for categorical columns
# Use 'data' directly to update it with one-hot encoded columns
data = pd.get_dummies(data, columns=categorical_cols, drop_first=True, dtype=int)

# Ensure all remaining boolean columns (from get_dummies) are int (0 or 1)
for col in data.select_dtypes(include='bool').columns:
    data[col] = data[col].astype(int)

print("Data preprocessing complete. First 5 rows of the processed data:")
display(data.head())

Data preprocessing complete. First 5 rows of the processed data:


,age,bp,sg,al,su,bgr,bu,sc,sod,pot,...,rc_6.4,rc_6.5,rc_8.0,rc_?,htn_yes,dm_yes,cad_yes,appet_poor,pe_yes,ane_yes
0,48.0,80.0,1.020,1.0,0.0,121.0,36.0,1.2,138.0,4.4,...,0,0,0,0,1,1,0,0,0,0
1,7.0,50.0,1.020,4.0,0.0,121.0,18.0,0.8,138.0,4.4,...,0,0,0,0,0,0,0,0,0,0
2,62.0,80.0,1.010,2.0,3.0,423.0,53.0,1.8,138.0,4.4,...,0,0,0,0,0,1,0,1,0,1
3,48.0,70.0,1.005,4.0,0.0,117.0,56.0,3.8,111.0,2.5,...,0,0,0,0,1,0,0,1,1,1
4,51.0,80.0,1.010,2.0,0.0,106.0,26.0,1.4,138.0,4.4,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
from sklearn.model_selection import KFold
import numpy as np
import shap # Ensure shap is imported
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier # Import KNeighborsClassifier
from sklearn.svm import SVC
import xgboost as xgb

kf = KFold(n_splits=5)

stability_scores = {}

for name, model in selected_models.items():

    fold_importance = []

    # Convert y_train_bal to a NumPy array for consistency with X_train_bal
    y_target_for_fit = y_train_bal.values

    for train_idx, val_idx in kf.split(X_train_bal): # X_train_bal is an ndarray after scaling

        # Pass NumPy arrays to model.fit
        model.fit(X_train_bal[train_idx], y_target_for_fit[train_idx])

        X_eval = X_train_bal[val_idx][:200] # Instances to explain

        # Choose appropriate explainer based on model type
        if isinstance(model, (RandomForestClassifier, GradientBoostingClassifier, xgb.XGBClassifier)):
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_eval)
            # For classification, shap_values is a list [shap_for_class_0, shap_for_class_1]
            # We need the values for the positive class (index 1)
            importance = abs(shap_values[1]).mean(axis=0)

        elif isinstance(model, (LogisticRegression, SVC, KNeighborsClassifier)): # Added KNeighborsClassifier
            # For models like LR, SVM, and KNN, use KernelExplainer
            # Use a sample from the current fold's validation set as background
            background_data = X_train_bal[val_idx][:200] # Same data as masker in original code
            explainer = shap.KernelExplainer(model.predict_proba, background_data)
            shap_values = explainer.shap_values(X_eval)
            # KernelExplainer's shap_values for classification is typically a list of arrays
            importance = abs(shap_values[1]).mean(axis=0)

        else:
            # Fallback for any other model types not explicitly handled
            # This case should ideally not be reached with the current `selected_models`
            # If it is reached, it indicates a model type not explicitly considered for SHAP
            print(f"Warning: Using generic shap.Explainer for {name} which might not be optimal.")
            explainer = shap.Explainer(model, X_eval)
            shap_values = explainer(X_eval)
            if isinstance(shap_values, list): # Check if it's a list (e.g., for classification)
                importance = abs(shap_values[1]).mean(axis=0)
            else:
                # For regression or other cases where shap_values is not a list
                importance = abs(shap_values.values).mean(axis=0)

        fold_importance.append(importance)

    fold_importance = np.array(fold_importance)

    # Using var() on an array of explanations to calculate stability
    stability = 1 / (1 + fold_importance.var()) # Using var() on an array of explanations

    stability_scores[name] = stability

print(stability_scores)

  0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/80 [00:00<?, ?it/s]